In [1]:
!pip -q install datasets pandas tqdm

import os
import re
import pandas as pd

from datasets import load_dataset
from tqdm.auto import tqdm

OUTPUT_FILE = "writing.csv"

if not os.path.exists(OUTPUT_FILE):
    pd.DataFrame(columns=["prompt", "response"]).to_csv(OUTPUT_FILE, index=False)

In [7]:
import os
import re
import html
import random
import pandas as pd

from tqdm.auto import tqdm
from datasets import load_dataset
from tokenizers import Tokenizer

random.seed(42)

tokenizer = Tokenizer.from_file(
    "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json"
)

OUTPUT_FILE = "writing.csv"

if os.path.exists(OUTPUT_FILE):
    writing_df = pd.read_csv(OUTPUT_FILE)
else:
    writing_df = pd.DataFrame(columns=["prompt", "response"])

existing_pairs = set(
    zip(
        writing_df["prompt"].astype(str),
        writing_df["response"].astype(str)
    )
)

rows = []

In [8]:
def clean_text(text):
    if text is None:
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [9]:
def token_count(text):
    return len(tokenizer.encode(text).ids)

In [10]:
def add_sample(prompt, response):
    prompt = clean_text(prompt)
    response = clean_text(response)

    if not prompt or not response:
        return

    if len(prompt.split()) < 4:
        return

    if len(response.split()) < 3:
        return

    total_tokens = token_count(prompt) + token_count(response)

    if total_tokens > 1024:
        return

    pair = (prompt, response)

    if pair in existing_pairs:
        return

    existing_pairs.add(pair)

    rows.append({
        "prompt": prompt,
        "response": response
    })

In [11]:
dataset = load_dataset(
    "EdinburghNLP/xsum",
    split="train"
)

In [12]:
summary_prompts = [
    "Summarize the following article.",
    "Write a concise summary.",
    "Summarize this passage.",
    "Summarize the article below.",
    "Provide a brief overview.",
    "Explain the article briefly.",
    "Capture the key points.",
    "Write a short abstract.",
    "Summarize the main ideas.",
    "What is this article about?",
    "Reduce this article to its essential points.",
    "Extract the important information.",
    "Provide a compact summary.",
    "Describe the article in a few sentences.",
    "Write a clear summary.",
    "Give the gist of the following article.",
    "Summarize the content below.",
    "Briefly describe the following article.",
    "Generate a summary.",
    "Summarize this document."
]

In [13]:
for sample in tqdm(dataset):

    document = clean_text(sample["document"])
    summary = clean_text(sample["summary"])

    if len(document.split()) < 80:
        continue

    prompt = f"{random.choice(summary_prompts)}\n\n{document}"

    add_sample(prompt, summary)

print(len(rows))

  0%|          | 0/204045 [00:00<?, ?it/s]

169005


In [14]:
preview = pd.DataFrame(rows)

preview.sample(
    min(10, len(preview)),
    random_state=42
)[["prompt", "response"]]

,prompt,response
146344,"Summarize the following article. Widely regarded as the third tier of the sport, after the NFL and college leagues, some matches are televised at prime time on Friday nights. But one school from Washington is struggling to get on the pitch, with three successive opponents forfeiting matches against them. The Wildcats from Archbishop Murphy High School (AMHS) won their first three matches this year by a combined score of 170-0. But their next three opponents - South Whidbey, Sultan and Granite Falls - have all refused to face them. AMHS have six players weighing at least 17 stone, including three at more than 21st, and are so dominant staff and parents from other schools are becoming increasingly concerned about player safety. A private Catholic school, AMHS is able to recruit students from a wider catchment area than the local comprehensive schools. ""The level of athletes they've been able to bring in on one team doesn't match up with a lot of the teams in our league,"" said Tim Dennis, head coach of Granite Falls. ""It's not that we're afraid to play the game, it's an injury issue."" During a tense meeting of parents, players and officials, the mother of one Granite Falls player said: ""My 14-year-old son is 5ft 8in and weighs 117 pounds (8st 5lbs). They've got 18-year-old players that are 6ft 5in and weigh 330 pounds (23st 5lbs). ""That's like putting a Volkswagen Bug against a truck."" But AMHS head coach Jerry Jensen said forfeiting matches did not ""ring true"" to what schools should be teaching their pupils. ""This is their opportunity to face adversity, power through it, and it will serve them well in their life,"" he said. AMHS play in the Cascade Conference, but there are growing calls for them to play in a higher division.","In some parts of the United States, high school American football is a huge deal."
38367,"What is this article about? It was the warmest December since records began in 1910 - and the wettest of any calendar month on record. Mean temperatures were about 4C (7.2F) above the long-term average. The Met Office says there is a direct link between the warmth and the record rains that brought widespread floods across Scotland, Northern Ireland and northern England. Stormy December 'was wettest on record' Storms propelled by the jet stream were mainly to blame, it says, with contributions from the El Nino weather phenomenon and man-made climate change. December was something of a freak month, it acknowledges. It says climate change has raised UK temperatures by around 1C (1.8F) so far, so it will be many decades before this level of extreme weather becomes the new winter norm. Other scientists say that with climate change, there will be no ""normal"" weather. Meanwhile, the unseasonal warmth has disrupted the natural world, and affected the way we live and the amount of cash we have to spend. People in T-shirts have been out in the gloom of winter, fuel bills are down, and the winter deaths total is likely to be lower than usual. Plants are also flowering at bizarre times. In Kew Gardens, in south-west London, a magnolia tree is in full bloom alongside daffodils. A currant bush - Ribes sanguineum - came into flower before Christmas when it is normally expected in March. My own garden in London still has a rose in flower and a strawberry on its bush. Tony Kirkham, head of the arboretum at Kew Gardens, said the recent jumbling of seasons was causing problems in the natural world. ""The plants are really mixed up, they don't know what season they're in. They think spring is on the way, and they need to flower and grow leaves to make food. ""The seasons are normally quite short so they do it as soon as time allows. ""The downside is is that we could get a frost, and all these young leaves are very tender and not used to temperatures below freezing, and they won't flower again in spring. ""And it's a food source for insects that won't be around when insects need it."" The Met Office 

In [15]:
df = pd.DataFrame(rows)

if len(df) > 8000:
    df = df.sample(
        n=8000,
        random_state=42
    ).reset_index(drop=True)

if os.path.exists(OUTPUT_FILE):

    old = pd.read_csv(OUTPUT_FILE)

    df = (
        pd.concat([old, df], ignore_index=True)
          .drop_duplicates(subset=["prompt", "response"])
          .reset_index(drop=True)
    )

df.to_csv(OUTPUT_FILE, index=False)

print(df.shape)

(16000, 2)


In [16]:
dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="train"
)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [17]:
summary_prompts = [
    "Summarize the following article.",
    "Write a concise summary of the article below.",
    "Provide a brief overview of the following news article.",
    "Summarize the main points.",
    "Write an abstract for the article.",
    "Condense the following article.",
    "Explain the article briefly.",
    "Capture the key ideas.",
    "Write a short summary.",
    "Summarize this document.",
    "What is this article about?",
    "Describe the article in a few sentences.",
    "Reduce this article to its essential points.",
    "Extract the important information.",
    "Provide a compact summary.",
    "Generate a brief summary.",
    "Write a news summary.",
    "Summarize the content below.",
    "Identify the main takeaway from the article.",
    "Summarize this text clearly."
]

start = len(rows)

for sample in tqdm(dataset):

    article = clean_text(sample["article"])
    summary = clean_text(sample["highlights"])

    if len(article.split()) < 80:
        continue

    prompt = f"{random.choice(summary_prompts)}\n\n{article}"

    add_sample(prompt, summary)

added = len(rows) - start

print(f"Added: {added:,}")

  0%|          | 0/287113 [00:00<?, ?it/s]

Added: 183,081


In [18]:
preview = pd.DataFrame(rows)

print(f"Current Samples: {len(preview):,}")

samples = preview.sample(min(10, len(preview)), random_state=42).reset_index(drop=True)

for i, row in samples.iterrows():
    print("=" * 120)
    print(f"Sample {i+1}")
    print("-" * 120)
    print("Prompt:")
    print(row["prompt"])
    print()
    print("Response:")
    print(row["response"])
    print()

Current Samples: 352,086
Sample 1
------------------------------------------------------------------------------------------------------------------------
Prompt:
Provide a brief overview. Warm temperatures are being experienced widely across Scotland, but so far they are still below the Scottish May record of 30.9C of five years ago. By 14:00, the temperature was 27.2C in Aboyne in Aberdeenshire, 25C in Inverness and Aviemore in the Highlands and 24C in Edinburgh and Glasgow. High temperatures have also been forecast for Friday when it could reach 29C. The top May temperature was recorded in Inverailort in the Highlands. BBC Scotland Weather said 27-28C was likely to be the hottest on Thursday, with the average expected to be between 13 and 16C. Warm and dry weather has been a feature of May this year. The previous month in Scotland was largely cold and wet with some snowfall. May is traditionally seen as one of the best months for long spells of fine weather in Scotland.

Response:
A

In [19]:
TARGET_SAMPLES = 8000

df = pd.DataFrame(rows)

if len(df) > TARGET_SAMPLES:
    df = (
        df.sample(n=TARGET_SAMPLES, random_state=42)
          .reset_index(drop=True)
    )

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)

    df = (
        pd.concat([existing, df], ignore_index=True)
          .drop_duplicates(subset=["prompt", "response"])
          .reset_index(drop=True)
    )

df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df):,} samples to {OUTPUT_FILE}")
print(df.sample(10, random_state=42))

Saved 23,786 samples to writing.csv
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [20]:
rows = []

In [21]:
dataset = load_dataset(
    "jhu-clsp/jfleg",
    split="validation"
)

README.md: 0.00B [00:00, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/141k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/755 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/748 [00:00<?, ? examples/s]

In [22]:
grammar_prompts = [
    "Correct the grammar in the following sentence.",
    "Fix the grammatical errors.",
    "Rewrite the sentence using proper grammar.",
    "Proofread the following text.",
    "Correct the spelling and grammar mistakes.",
    "Improve the fluency of the sentence.",
    "Rewrite the following sentence naturally.",
    "Edit the sentence for grammatical correctness.",
    "Polish the writing while preserving its meaning.",
    "Improve the readability of the following sentence.",
    "Rewrite the sentence to sound more natural.",
    "Revise the following text for grammar and clarity.",
    "Correct the writing errors without changing the meaning.",
    "Improve the sentence structure.",
    "Make the following sentence grammatically correct.",
    "Rewrite this text using standard English.",
    "Refine the grammar of the following sentence.",
    "Correct any language mistakes in the text.",
    "Edit the sentence for better fluency.",
    "Rewrite the sentence with improved grammar."
]

start = len(rows)

for sample in tqdm(dataset):

    source = clean_text(sample["sentence"])

    for target in sample["corrections"]:

        target = clean_text(target)

        if source == target:
            continue

        prompt = f"{random.choice(grammar_prompts)}\n\n{source}"

        add_sample(prompt, target)

added = len(rows) - start

print(f"Added: {added:,}")

  0%|          | 0/755 [00:00<?, ?it/s]

Added: 2,567


In [23]:
preview = pd.DataFrame(rows)

print(f"Samples: {len(preview):,}")

preview.sample(10, random_state=42)[["prompt", "response"]]

Samples: 2,567


,prompt,response
973,Fix the grammatical errors. In both advertisements is said that these tooth pastes will make your teeth briliant and brighter .,Both advertisements say that the toothpaste will make your teeth brilliant and brighter .
2081,Correct the grammar in the following sentence. an 7unsuspection user cannto tell the entruy has vees tampered with .,An unsuspecting user cannot tell the century has been tampered with .
2017,"Improve the sentence structure. Really successful people gained their fortune by doing somethig new , something that no one else has ever done before , that means that nobody knew how to do it well that was their risk .","Really successful people gained their fortunes by trying something new , something that no one else has ever done before , and taking risks where people were too afraid to ."
952,"Polish the writing while preserving its meaning. For example , they do not like to wait much .","For example , they do not like to wait ."
2531,"Correct the writing errors without changing the meaning. But if one who majors in art also learns something about natural science , he will be lucky to be equipped with the ability to think more logically .","But if one who majors in art also learns something about natural science , he will have the fortune of gaining the ability to think more logically ."
1670,"Rewrite this text using standard English. It is also undeniable that if a man has diversifed knowledge , it would be immediate use .","It is also undeniable that if a man has diversified knowledge , it would be used immediately ."
2122,Rewrite the sentence to sound more natural. I had lots of problem about studying science .,I had lots of problems when studying science .
178,Make the following sentence grammatically correct. In today 's Compuer skill is first important life skill .,"Today , computer skills are the most important life skill ."
1421,"Revise the following text for grammar and clarity. These useful skills that I learned from reality are going to be the greatest gift for my future career , and they can not be found in the text books anyways .","These useful skills that I have learned from real life are going to be the greatest gift to my future career , and they are skills that cannot be found in a textbook ."
2265,Polish the writing while preserving its meaning. They bring us to far places in quite a fast speed without much human physical exertion .,They bring us to far away places quite quickly and without much human physical exertion .


In [24]:
TARGET_SAMPLES = 7000

df = pd.DataFrame(rows)

if len(df) > TARGET_SAMPLES:
    df = (
        df.sample(n=TARGET_SAMPLES, random_state=42)
          .reset_index(drop=True)
    )

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)

    df = (
        pd.concat([existing, df], ignore_index=True)
          .drop_duplicates(subset=["prompt", "response"])
          .reset_index(drop=True)
    )

df.to_csv(OUTPUT_FILE, index=False)

rows = []

print(f"Writing Dataset Size: {len(df):,}")

Writing Dataset Size: 26,353


In [28]:
dataset = load_dataset(
    "billsum",
    split="train"
)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/91.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

data/ca_test-00000-of-00001.parquet:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

In [29]:
billsum_prompts = [
    "Summarize the following bill.",
    "Write a concise summary of the legislation below.",
    "Provide a brief overview of the following bill.",
    "Explain the purpose of this bill.",
    "Summarize the key provisions of the following legislation.",
    "Describe the main objectives of this bill.",
    "Write a short summary of the legislative text.",
    "Summarize the following legal document.",
    "Extract the main points from this bill.",
    "Provide a compact summary of the legislation.",
    "What is the purpose of this bill?",
    "What does the following bill propose?",
    "Summarize the important sections of this bill.",
    "Explain this legislative document briefly.",
    "Write an abstract for the following bill.",
    "Summarize the following government document.",
    "Describe the key ideas presented in this bill.",
    "Write a clear summary of the legal text.",
    "Summarize the content below.",
    "Identify the main purpose of the following legislation."
]

start = len(rows)

for sample in tqdm(dataset):

    document = clean_text(sample["text"])
    summary = clean_text(sample["summary"])

    if len(document.split()) < 50:
        continue

    prompt = f"{random.choice(billsum_prompts)}\n\n{document}"

    add_sample(prompt, summary)

added = len(rows) - start

print(f"Added: {added:,}")

  0%|          | 0/18949 [00:00<?, ?it/s]

Added: 506


In [30]:
preview = pd.DataFrame(rows)

print(f"Samples: {len(preview):,}")

preview.sample(10, random_state=42)[["prompt", "response"]]

Samples: 506


prompt  \
173  Describe the main objectives of this bill. SECTION 1. SHORT TITLE. This Act may be cited as the ``Reasserting American Leadership in Space Act'' or the ``REAL Space Act''. SEC. 2. FINDINGS. Congress finds the following: (1) The 109th Congress passed the National Aeronautics and Space Administration Authorization Act of 2005 overwhelmingly, establishing as the National Aeronautics and Space Administration's priority human space flight goal: ``To develop a sustained human presence on the Moon . . . to promote exploration, commerce, science, and United States preeminence in space as a stepping stone for the future exploration of Mars and other destinations.''. (2) The 110th Congress overwhelmingly reaffirmed the vision of returning to the Moon as an integral part of exploring further into our solar system through the passage of the National Aeronautics and Space Administration Authorization Act of 2008, expressing support for ``the broad goals of the space exploration policy of the United States, including the eventual return to and exploration of the Moon and other destinations in the solar system and the important national imperative of independent access to space''. (3) The 111th Congress, in the National Aeronautics and Space Administration Authorization Act of 2010, called for the development of a heavy lift capability of greater than 130 metric tons consisting of the Space Launch System (SLS) and Multi-Purpose Crew Vehicle (MPCV) to pursue exploration, yet fell short on explicitly stating a clear destination. (4) The 112th Congress has reaffirmed this commitment to the development of a heavy lift capability. (5) A sustained human presence on the Moon will allow astronauts and researchers the opportunity to leverage new technologies in addressing the challenges of sustaining life on another celestial body, lessons which are necessary and applicable as we explore further into our solar system, to Mars and beyond. (6) A sustained human presence on the Moon would once again inspire and engage public interest in our space program, motivating young people to excel in the vital subjects of math and science, subjects in which American students lag behind our international competitors. (7) A sustained human presence on the Moon would challenge American industry to continue to develop technologies that not only enhance our exploration programs but can be applied across all disciplines of science. (8) The commercial applications of space technologies have had tens of billions of dollars in economic impact, including products from semiconductors and aircraft controls to scratch- resistant lenses and water purification systems. (9) The healthcare technologies derived from our space program, such as the portable x-ray machine, the MRI, advanced life-saving diagnostics, and the implantable heart aid, have saved and improved countless lives. (10) Space is the world's ultimate high ground, returning to the Moon and reinvigorating our human space flight program is a matter of national security. (11) Technologies developed and sustained by the National Aeronautics and Space Administration's human space flight program, such as liquid and solid rocket propulsion, environmental and life support systems, and communications, navigation, and control systems are important to our military. (12) China and Russia, understanding the economic and strategic importance of human space flight, have declared their intentions of colonizing the Moon and are advancing their lunar exploration plans. (13) It is strategically important that the United States possess and maintain the capabilities of unfettered operation in the space domain, and not cede the space domain to other nations. SEC. 3. MISSION. In accordance with the National Aeronautics and Space Administration Authorization Act of 2005, which established as the National Aeronautics and Space Administration's priority goal: ``To develop a sustained human presence on the Moon . . . to promot

In [31]:
TARGET_SAMPLES = 5000

df = pd.DataFrame(rows)

if len(df) > TARGET_SAMPLES:
    df = (
        df.sample(n=TARGET_SAMPLES, random_state=42)
          .reset_index(drop=True)
    )

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)

    df = (
        pd.concat([existing, df], ignore_index=True)
          .drop_duplicates(subset=["prompt", "response"])
          .reset_index(drop=True)
    )

df.to_csv(OUTPUT_FILE, index=False)

rows = []

print(f"Writing Dataset Size: {len(df):,}")

Writing Dataset Size: 26,859
